### vLLM for efficient serving

In [1]:
from vllm import LLM, SamplingParams

INFO 04-12 03:06:17 [__init__.py:239] Automatically detected platform cuda.


In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
#serve vllm thorugh openai api servers & make concurent calls

In [4]:
#!vllm serve lora/lora_16bit_merged_3b_r128_s1000_i1000_v1 --max-model-len=1024 --dtype auto --api-key my-api-key

### OpenAI style client

In [5]:
from openai import OpenAI
# Set OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "my-api-key"
openai_api_base = "http://localhost:8000/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)


def get_reponse(description):
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. 
        Write a response that appropriately completes the request.

                ### Instruction:
                Please write a SVG code for the given input.

                ### Input:
                {}

                ### Response:
                """
    
    formatted_input = alpaca_prompt.format(description)
    chat_response = client.chat.completions.create(
        model="lora/lora_16bit_merged_3b_r128_s1000_i1000_v1",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"{formatted_input}"},
        ]
    )
    return chat_response.choices[0].message.content


In [6]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/svg_score_test.csv',header=[0])
#df=df[['description','svg']]

### Sequentional execution (GPU time: 380s)

In [8]:
# import time
# from tqdm import tqdm

# tqdm.pandas()  # Enable tqdm for pandas apply

# start_time = time.time()

# df['response'] = df['description'].progress_apply(lambda x: get_reponse(x))

# end_time = time.time()
# print(f"Total time taken: {end_time - start_time:.2f} seconds")

### Concurrent execution

In [8]:
import time
from tqdm import tqdm
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# Wrap tqdm over futures
def parallel_apply_with_tqdm(func, data, max_workers=12):
    results = [None] * len(data)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(func, data[i]): i for i in range(len(data))}
        for future in tqdm(as_completed(futures), total=len(data)):
            idx = futures[future]
            try:
                results[idx] = future.result()
            except Exception as e:
                results[idx] = None
                print(f"Error at index {idx}: {e}")
    return results

# Example usage
start_time = time.time()

df['response'] = parallel_apply_with_tqdm(get_reponse, df['description'].tolist(), max_workers=12)

end_time = time.time()
print(f"Total time taken: {end_time - start_time:.2f} seconds")


  0%|                                                    | 0/76 [00:00<?, ?it/s]

INFO 04-12 03:07:50 [chat_utils.py:379] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
INFO 04-12 03:07:50 [logger.py:39] Received request chatcmpl-474cf6f657fa436c9c103cac5cee5e09: prompt: "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 12 Apr 2025\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nBelow is an instruction that describes a task, paired with an input that provides further context. \n        Write a response that appropriately completes the request.\n\n                ### Instruction:\n                Please write a SVG code for the given input.\n\n                ### Input:\n                 'Vibrant autumn forest',\n\n                ### Response:<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty

  1%|▌                                           | 1/76 [00:02<03:01,  2.42s/it]

INFO:     127.0.0.1:50472 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO 04-12 03:07:52 [logger.py:39] Received request chatcmpl-ce2638ecabdf49afa6ba5a26a9ed17d1: prompt: "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 12 Apr 2025\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nBelow is an instruction that describes a task, paired with an input that provides further context. \n        Write a response that appropriately completes the request.\n\n                ### Instruction:\n                Please write a SVG code for the given input.\n\n                ### Input:\n                 'Snow-capped mountains',\n\n                ### Response:<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n", params: SamplingParams(n=1, presence_penalty=0.0, frequency_penalty=0.0, repetition_penalty=1.0, temperature=0.6, top_p=0.9, top_k=-1, min_p=0.0, seed=None, stop=[], stop_token_ids

  3%|█▏                                          | 2/76 [00:03<02:13,  1.81s/it]

...

 97%|█████████████████████████████████████████▊ | 74/76 [00:37<00:01,  1.68it/s]

INFO:     127.0.0.1:50530 - "POST /v1/chat/completions HTTP/1.1" 200 OK
INFO:     127.0.0.1:50526 - "POST /v1/chat/completions HTTP/1.1" 200 OK


100%|███████████████████████████████████████████| 76/76 [00:39<00:00,  1.93it/s]

INFO:     127.0.0.1:50488 - "POST /v1/chat/completions HTTP/1.1" 200 OK
Total time taken: 39.42 seconds


INFO 04-12 03:08:36 [loggers.py:80] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 65.6 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 83.1%
INFO 04-12 03:08:46 [loggers.py:80] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 83.1%
INFO 04-12 03:08:56 [loggers.py:80] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 83.1%
INFO 04-12 03:09:06 [loggers.py:80] Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 0.0 tokens/s, Running: 0 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.0%, Prefix cache hit rate: 83.1%


In [9]:
import os
import signal

# Pause the process (send SIGSTOP)
os.kill(vllm_process.pid, signal.SIGSTOP)
print("vLLM server is paused.")


vLLM server is paused.


In [11]:
vllm_process.pid

30789

In [12]:
os.kill(vllm_process.pid, signal.SIGKILL)

In [ ]:
# import threading
# import queue
# import gc
# from vllm import LLM, SamplingParams

# class Model:
#     def __init__(self):
#         self.model_path = "./lora/Llama-3.2-3B-Instruct_r256_s2000_i1000_v1"
#         self.model = LLM(
#             model=self.model_path,
#             max_model_len=1024,
#             gpu_memory_utilization=0.85,
#             dtype="half",
#             seed=123,
#             disable_log_stats=True
#         )

#         self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
#         self.constraints = svg_constraints.SVGConstraints()
#         self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

#         self.task_queue = queue.Queue()
#         self.result_dict = {}
#         self.lock = threading.Lock()

#         # Start the background worker
#         self.worker_thread = threading.Thread(target=self._worker, daemon=True)
#         self.worker_thread.start()

#     def _worker(self):
#         while True:
#             task_id, description = self.task_queue.get()
#             try:
#                 svg = self._predict_single(description)
#                 with self.lock:
#                     self.result_dict[task_id] = svg
#             except Exception as e:
#                 with self.lock:
#                     self.result_dict[task_id] = str(e)
#             self.task_queue.task_done()

#     def _predict_single(self, description: str) -> str:
#         alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. 
#         Write a response that appropriately completes the request.
            
#                 ### Instruction:
#                 Please write a SVG code for the given input.
            
#                 ### Input:             
#                 {}
            
#                 ### Response:
#                 """
#         formatted_input = alpaca_prompt.format(description)
#         sampling_params = SamplingParams(temperature=0.6, top_p=0.95, max_tokens=1024)
#         outputs = self.model.generate([formatted_input], sampling_params)

#         for output in outputs:
#             generated_text = output.outputs[0].text
#         base_svg_code = SVGProcessor.clean_and_extract_svgs(generated_text, self.default_svg)
#         clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)
#         return SVGProcessor.svg_conversion_check(description, clean_svg_code, self.default_svg)

#     def predict(self, description: str) -> str:
#         task_id = id(description) + threading.get_ident()  # Unique task ID
#         self.task_queue.put((task_id, description))
#         while True:
#             with self.lock:
#                 if task_id in self.result_dict:
#                     result = self.result_dict.pop(task_id)
#                     return result

#     def close_model(self):
#         del self.model
#         gc.collect()
